In [ ]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "esol":["0205_ALL/0205esol", "0205_ALL/Vanilla_Reg"],
                  "tox21":["0205_ALL/0205Tox",
                             "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type} (\d)
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold,architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold,architecture, model_type = match.groups()
               
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_paths = [
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_train.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_validation.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_test.csv")
                    ]
                    dfs_correct = []
                    dfs_incorrect = []
                    for file in file_paths:
                        try:
                            
                            df = pd.read_csv(file)
                            if 'original_logit_vanilla' not in df.columns:
                                print(file_paths)
                                continue
                            if dataset_name != "tox21":
                                df.rename(columns={'original_logit_class_0': 'original_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_0': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_vanilla': 'new_logit_vanilla'}, inplace=True)
                                df.rename(columns={'sigmoid_importance_for_class_0': 'sigmoid_importance'}, inplace=True)
                        except Exception as e:
                            print(f"Error reading {file}: {e}")
                            continue

                        
                        if dataset_name in ['Lipophilicity','esol']:
                        
                            # Define a tolerance (this value should be chosen based on your problem's scale)
                            tolerance = 0.3  # Adjust as needed

                            # Compute the absolute error for each prediction.
                            df['error'] = np.abs(df['original_logit'] - df['class_label'])

                            # Split the data based on whether the prediction error is within the acceptable tolerance.
                            df_correct = df[df['error'] < tolerance]
                            df_incorrect = df[df['error'] >= tolerance]

                            # Optionally, if you want to inspect the computed errors:
                            print("Mean error for correct predictions:", df_correct['error'].mean())
                            print("Mean error for incorrect predictions:", df_incorrect['error'].mean())
                        else:
                            # Compute prediction (modify as needed)
                            df['predicted'] = (df['original_logit'] > 0).astype(int)

                            # Split data based on correct vs. incorrect predictions
                            df_correct = df[df['predicted'] == df['class_label'].astype(int)]
                            df_incorrect = df[df['predicted'] != df['class_label'].astype(int)]
                        
                        dfs_correct.append(df_correct)
                        dfs_incorrect.append(df_incorrect)
                    
                    # Only process further if we have at least one correct sample dataframe.
                    if dfs_correct:
                        df_correct = pd.concat(dfs_correct, ignore_index=True)
                    else:
                        continue
                    if dfs_incorrect:
                        df_incorrect = pd.concat(dfs_incorrect, ignore_index=True)
                    else:
                        df_incorrect = pd.DataFrame()  # no incorrect samples
                    
                    # --- MOTIF FREQUENCY FILTERING ---
                    # Count the occurrence of each motif in the correct examples.
                    # It is assumed that there is a column named 'motif_id' in the data.
                    motif_counts = df_correct['motif'].value_counts()
                    # Filter to include only rows where the motif frequency is greater than the threshold.
                    df_correct = df_correct[df_correct['motif'].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                    # Create bins for sigmoid_importance (10 bins)
                    bin_edges = np.linspace(0, 1, 11)
                    # --- CALCULATE ABSOLUTE LOGIT DIFFERENCE ---
                    
                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_correct['logit_diff'] = np.abs(Sigmoid(df_correct['original_logit']) - Sigmoid(df_correct['new_logit']))
                    df_correct['logit_diff_vanilla'] = np.abs(Sigmoid(df_correct['original_logit_vanilla']) - Sigmoid(df_correct['new_logit_vanilla']))
                    df_correct['sigmoid_bin'] = pd.cut(df_correct['sigmoid_importance'],
                                                           bins=bin_edges, include_lowest=True)
                    
                    # Tag the data (if you want to keep track in the DataFrame)
                    df_correct = df_correct.assign(Correct='Correct')
                    
                    
                    # --- PROCESS INCORRECT SAMPLES IF REQUESTED ---
                    if include_incorrect and not df_incorrect.empty:
                        # Filter incorrect samples by motif frequency.
                        motif_counts_incorrect = df_incorrect['motif'].value_counts()
                        df_incorrect = df_incorrect[df_incorrect['motif'].map(motif_counts_incorrect) > motif_frequency_threshold[dataset_name]]
                        # if dataset_name == 'tox21':
                        # Calculate absolute logit difference for incorrect samples.
                        df_incorrect['logit_diff'] = np.abs(Sigmoid(df_incorrect['original_logit']) - 
                                                           Sigmoid(df_incorrect['new_logit']))
                        df_incorrect['logit_diff_vanilla'] = np.abs(Sigmoid(df_incorrect['original_logit_vanilla']) - Sigmoid(df_incorrect['new_logit_vanilla']))
                        # Create bins for sigmoid_importance (10 bins)
                        df_incorrect['sigmoid_bin'] = pd.cut(df_incorrect['sigmoid_importance'],
                                                                 bins=bin_edges, include_lowest=True)
                        
                        # Tag as incorrect.
                        df_incorrect = df_incorrect.assign(Correct='Incorrect')
                        
                        # Combine correct and incorrect samples.
                        df_combined = pd.concat([df_correct, df_incorrect], ignore_index=True)
                    else:
                        df_combined = df_correct
                    
                    
                    # Update the counts after filtering.
                    num_correct = len(df_correct)
                    num_incorrect = len(df_incorrect)
                    print(f"Dataset: {dataset_name}, Architecture: {architecture}, Fold: {fold}, Type: {model_type} -- Correct Samples (after motif filter): {num_correct}, Incorrect Samples: {num_incorrect}")
                    

                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_combined)

# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )

# ================================
# PLOTTING: ONE PDF, EACH PAGE IS ONE ARCHITECTURE
# ================================
# Here the rows of the grid are datasets and the columns are unique types.
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = types  # e.g., ["RBRICS", "MGSSL"]

# Determine grid dimensions:
n_datasets = len(unique_datasets)
n_arch = len(architectures)
n_cols_total = n_datasets * 2  # Two sub-columns per dataset, one per type.
n_rows_total = n_arch  # Rows correspond to architectures.

# Initialize legend handles & labels
legend_handles, legend_labels = None, None

for model_type in unique_types:
    # Determine grid dimensions: datasets as rows, architectures as columns.
    n_rows = len(unique_datasets)
    n_cols = len(architectures)
    
    # Create a figure with appropriate size.
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 8, n_rows * 6), squeeze=True)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.25, hspace=0.35)
    
    legend_handles, legend_labels = None, None  # To capture legend once
    
    # Populate each subplot.
    for d_idx, dataset in enumerate(unique_datasets):
        for a_idx, arch in enumerate(architectures):
            ax = axes[d_idx, a_idx]
            # Retrieve data for this combination.
            data = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, pd.DataFrame())
            
            if not data.empty:
                # Plotting code (similar to original).
                counts = data['sigmoid_bin'].value_counts().sort_index()
                counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()
                # Calculate average for each motif within each sigmoid_bin
                # pdb.set_trace()
                motif_avg = data.groupby(["motif", "sigmoid_bin"])[["logit_diff_vanilla", "logit_diff"]].mean().reset_index()

                # Melt the dataframe for easier plotting
                data_melted = motif_avg.melt(
                    id_vars=["motif", "sigmoid_bin"], 
                    value_vars=["logit_diff_vanilla", "logit_diff"], 
                    var_name="metric", 
                    value_name="value"
                )

                # data_melted = pd.melt(
                #     data, 
                #     id_vars=['sigmoid_bin'], 
                #     value_vars=['logit_diff_vanilla', 'logit_diff'], 
                #     var_name='metric', 
                #     value_name='value'
                # )
                palette = {"logit_diff": "#d62728", "logit_diff_vanilla": "#1f77b4"}
                
                sns.boxplot(
                    ax=ax,
                    x='sigmoid_bin',
                    y='value',
                    hue='metric',
                    palette=palette,
                    data=data_melted,
                    width=0.6,
                    dodge=True,
                    linewidth=1.5,
                )
                
                # Capture legend handles once.
                if legend_handles is None:
                    legend_handles, legend_labels = ax.get_legend_handles_labels()
                ax.get_legend().remove()
                
                # Add histogram for percentage counts.
                if not counts_percentage.empty:
                    ax.bar(
                        x=np.arange(len(counts_percentage)),
                        height=-counts_percentage.values,
                        color='tab:orange',
                        alpha=0.5,
                        width=0.4,
                        align='center'
                    )
                
                # Configure axes.
                ax.set_ylim(-0.6)
                ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
                ax.set_yticks([-0.5, -0.25, 0, 0.5])
                ax.set_yticklabels(['50', '25', '0', '0.5'])
                ax.set_xlabel('Normalized Motif Importance', fontsize=14, fontweight='bold')
                ax.tick_params(axis='x', rotation=45)
                
                # Set y-label for the first column.
                if a_idx == 0:
                    ax.set_ylabel(f' % Count     |     Abs Prob Diff', fontweight='bold', fontsize=12)
                else:
                    ax.set_ylabel('')
            else:
                ax.set_visible(False)
    
    # Add dataset labels on the left.
    for d_idx, dataset in enumerate(unique_datasets):
        pos = axes[d_idx, 0].get_position()
        fig.text(pos.x0 - 0.05, pos.y0 + pos.height/2, dataset, 
                 ha='right', va='center', rotation=90, fontsize=22, fontweight='bold')
    
    # Add architecture labels on top.
    for a_idx, arch in enumerate(["GAT", "GCN", "GIN"]):
        pos = axes[0, a_idx].get_position()
        fig.text(pos.x0 + pos.width/2, pos.y1 + 0.02, arch, 
                 ha='center', va='bottom', fontsize=22, fontweight='bold')
    
    # Add global legend.
    if legend_handles and legend_labels:
        fig.legend(
            legend_handles, 
            ['Vanilla', 'MOSE'], 
            title='Model', 
            loc='upper right', 
            bbox_to_anchor=(0.98, 0.93),  # Moves the legend to the top-right
            ncol=2,  # Keep items in a single row
            fontsize=14, 
            title_fontsize=16, 
            frameon=True, 
            edgecolor='black'
        )
    
    # Save the figure.
    plt.savefig(f'{model_type}_{fold}_comparison.png', bbox_inches='tight')
    plt.close()
print("Saved combined plots as combined_plots.png")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re

# Set style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Frequency thresholds
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "Benzene":70,
    "Alkane_Carbonyl":50,
    "Fluoride_Carbonyl":50,
    "hERG": 90,
    "tox21": 70,
}

# Replace with your root path
folder_name = "RBRICSwithMinority"

root_dirs_dict = {
    "Mutagenicity": f"{folder_name}/MOSE_BC",
    "hERG": f"{folder_name}/MOSE_BC",
    "BBBP": f"{folder_name}/MOSE_BC",
    "esol": f"{folder_name}/MOSE_Reg",
    "Lipophilicity": f"{folder_name}/MOSE_Reg",
    "Benzene": f"{folder_name}/MOSE_BC",
    "Alkane_Carbonyl": f"{folder_name}/MOSE_BC",
    "Fluoride_Carbonyl": f"{folder_name}/MOSE_BC",
}

architectures = {"GAT", "GCN", "GIN", "SAGE"}
types = {"RBRICS", "None"}

folder_pattern = re.compile(
    r"EXPT-\d+[A-Z]*-([\w_]+)-SEED-\d+-FOLD-(\d+)-(\w+)-EXPLLR0\.01-[\w\d]+-(\w+)$"
)
folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[A-Z]*-([\w_]+)-SEED-\d+-FOLD-(\d+)-(\w+)-None-None-(\w+)$"
)

# Master data collection
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for dataset_name, root_dir in root_dirs_dict.items():
    for folder in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder)

        # Folder pattern match
        match = folder_pattern.match(folder) or folder_pattern_vanilla.match(folder)
        if not match:
            continue
        name_of_dataset, fold, architecture, model_type = match.groups()
        if name_of_dataset != dataset_name or architecture not in architectures or model_type not in types:
            continue

        # Gather all CSV files in the directory
        file_paths = [
            os.path.join(folder_path, f)
            for f in os.listdir(folder_path)
            if f.endswith(".csv")
        ]
        dfs_all = []

        for file in file_paths:
            try:
                df = pd.read_csv(file)
            except Exception as e:
                print(f"Error reading {file}: {e}")
                continue

            # Motif frequency filtering
            motif_counts = df["motif"].value_counts()
            df = df[df["motif"].map(motif_counts) > motif_frequency_threshold[dataset_name]]

            # Bin and logit diff
            df["logit_diff"] = np.abs(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
            df["diff_sign"] = np.sign(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
            df["sigmoid_bin"] = pd.cut(df["sigmoid_importance"], bins=np.linspace(0, 1, 11), include_lowest=True)
            dfs_all.append(df)

        if not dfs_all:
            continue

        df_all = pd.concat(dfs_all, ignore_index=True)
        

        plotting_data[architecture][dataset_name][model_type].append(df_all)

        print(f"[{dataset_name}] {architecture}-{model_type} | Fold {fold} | Samples: {len(df_all)}")

# Final merge over folds
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )


# RBRICSwithMinority

In [1]:
import os
import re
import pandas as pd
import numpy as np
from collections import defaultdict

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

folder_name = "../RBRICS_MINORITY_MP2_DIM16_CORRECTED"
root_dirs_dict = {
    "Mutagenicity": f"{folder_name}/MOSE_BC",
    "hERG": f"{folder_name}/MOSE_BC",
    "BBBP": f"{folder_name}/MOSE_BC",
    "esol": f"{folder_name}/MOSE_Reg",
    "Lipophilicity": f"{folder_name}/MOSE_Reg",
    "Benzene": f"{folder_name}/MOSE_BC",
    "Alkane_Carbonyl": f"{folder_name}/MOSE_BC",
    "Fluoride_Carbonyl": f"{folder_name}/MOSE_BC",
}

expl_lr_values = ["0001", "001", "01"]
gnn_lr_values = ["0001", "001", "01"]
architectures = {"GAT", "GCN", "GIN", "SAGE"}
types = {"RBRICS"}
valid_folds = [str(f) for f in range(5)]

motif_frequency_threshold = {
    "esol": 10, "BBBP": 18, "Lipophilicity": 38, "Mutagenicity": 70,
    "Benzene": 70, "Alkane_Carbonyl": 50, "Fluoride_Carbonyl": 50,
    "hERG": 90, "tox21": 70,
}

# Updated to include expl_lr and gnn_lr
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list)))))
training_losses = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
validation_losses = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))
found_keys = set()
missing_data_records = []

for EXPL_LR in expl_lr_values:
    for GNN_LR in gnn_lr_values:
        print(f"\n=== Checking EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR} ===")
        folder_pattern = re.compile(
            fr"EXPT-\d+[A-Z]*-([\w_]+)-SEED-\d+-FOLD-(\d+)-(\w+)-EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-([\w\d]+|None)-(RBRICS|None)$"
        )

        for dataset_name, root_dir in root_dirs_dict.items():
            for fold in valid_folds:
                for arch in architectures:
                    for model_type in types:
                        folder_found = False
                        missing_files = []
                        dfs_all = []

                        if not os.path.exists(root_dir):
                            missing_files.append(f"Root directory missing: {root_dir}")
                        else:
                            for folder in os.listdir(root_dir):
                                match = folder_pattern.match(folder)
                                if not match:
                                    continue

                                dset, f, a, _, t = match.groups()
                                if (dset == dataset_name and f == fold
                                        and a == arch and t == model_type):
                                    folder_found = True
                                    folder_path = os.path.join(root_dir, folder)
                                    training_path = os.path.join(folder_path, "explainer", f"{dataset_name}.csv")
                                    result_files = [
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_train.csv"),
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_validation.csv"),
                                        os.path.join(folder_path, f"{dataset_name}_explanation_result_with_test.csv"),
                                    ]

                                    # Check for missing files
                                    if not os.path.exists(training_path):
                                        missing_files.append(training_path)
                                    for file in result_files:
                                        if not os.path.exists(file):
                                            missing_files.append(file)

                                    if not missing_files:
                                        # Read training/validation losses
                                        try:
                                            df_train = pd.read_csv(training_path)
                                            if "Train Loss" in df_train.columns and "Val Loss" in df_train.columns:
                                                training_losses[dataset_name][arch][EXPL_LR][GNN_LR].append(df_train["Train Loss"].values)
                                                validation_losses[dataset_name][arch][EXPL_LR][GNN_LR].append(df_train["Val Loss"].values)
                                        except Exception as e:
                                            print(f"Error reading {training_path}: {e}")
                                            missing_files.append(f"Error reading training CSV: {e}")

                                        # Read and process result CSVs
                                        for file in result_files:
                                            try:
                                                df = pd.read_csv(file)
                                                motif_counts = df["motif"].value_counts()
                                                df = df[df["motif"].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                                                df["logit_diff"] = np.abs(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                                                df["diff_sign"] = np.sign(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                                                df["sigmoid_bin"] = pd.cut(df["sigmoid_importance"], bins=np.linspace(0, 1, 11), include_lowest=True)
                                                dfs_all.append(df)
                                            except Exception as e:
                                                print(f"Error reading {file}: {e}")
                                                missing_files.append(f"Error reading {file}: {e}")
                                    break  # Stop after finding matching folder

                        if not folder_found:
                            missing_files.append("Folder matching pattern not found")

                        if missing_files:
                            missing_data_records.append({
                                "Dataset": dataset_name,
                                "Architecture": arch,
                                "Type": model_type,
                                "Fold": fold,
                                "EXPL_LR": f"0.{EXPL_LR}",
                                "GNN_LR": f"0.{GNN_LR}",
                                "Missing Files": ", ".join(missing_files)
                            })
                            print(f"Missing: Dataset={dataset_name}, Arch={arch}, Type={model_type}, Fold={fold}, "
                                  f"EXPL_LR=0.{EXPL_LR}, GNN_LR=0.{GNN_LR}")
                        else:
                            # If all files exist, add to found_keys and plotting_data
                            found_keys.add((dataset_name, arch, model_type, fold, EXPL_LR, GNN_LR))
                            if dfs_all:
                                df_all = pd.concat(dfs_all, ignore_index=True)
                                plotting_data[arch][dataset_name][model_type][EXPL_LR][GNN_LR].append(df_all)
                                print(f"Collected: [{dataset_name}] {arch}-{model_type} Fold={fold} EXPL_LR=0.{EXPL_LR} GNN_LR=0.{GNN_LR} Samples={len(df_all)}")

# Merge folds for each configuration in plotting_data
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            for EXPL_LR in plotting_data[arch][dataset][model_type]:
                for GNN_LR in plotting_data[arch][dataset][model_type][EXPL_LR]:
                    plotting_data[arch][dataset][model_type][EXPL_LR][GNN_LR] = pd.concat(
                        plotting_data[arch][dataset][model_type][EXPL_LR][GNN_LR], ignore_index=True
                    )

# Save missing file report to CSV
report_df = pd.DataFrame(missing_data_records)
report_csv_path = "missing_files_report.csv"
report_df.to_csv(report_csv_path, index=False)
print(f"\n==== Missing Files Report saved to {report_csv_path} ====")



=== Checking EXPLLR=0.0001, GNNLR=0.0001 ===
Collected: [Mutagenicity] GAT-RBRICS Fold=0 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] SAGE-RBRICS Fold=0 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GIN-RBRICS Fold=0 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GCN-RBRICS Fold=0 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16504
Collected: [Mutagenicity] GAT-RBRICS Fold=1 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] SAGE-RBRICS Fold=1 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GIN-RBRICS Fold=1 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GCN-RBRICS Fold=1 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16810
Collected: [Mutagenicity] GAT-RBRICS Fold=2 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16382
Collected: [Mutagenicity] SAGE-RBRICS Fold=2 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=16382
Collected: [Mutagenicity] GIN-RBRICS Fold=2 EXPL_LR=0.0001 GNN_LR=0.0001 Samples=

In [3]:
import os
import re
import pandas as pd
import numpy as np
import seaborn as sns
from collections import defaultdict

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

def Sigmoid(x):
    return 1 / (1 + np.exp(-x))

folder_name = "RBRICS_MINORITY_MP2_DIM16_CORRECTED"
root_dirs_dict = {
    "Mutagenicity": f"{folder_name}/MOSE_BC",
    "hERG": f"{folder_name}/MOSE_BC",
    "BBBP": f"{folder_name}/MOSE_BC",
    "esol": f"{folder_name}/MOSE_Reg",
    "Lipophilicity": f"{folder_name}/MOSE_Reg",
    "Benzene": f"{folder_name}/MOSE_BC",
    "Alkane_Carbonyl": f"{folder_name}/MOSE_BC",
    "Fluoride_Carbonyl": f"{folder_name}/MOSE_BC",
}
EXPL_LR = "0001" # or "001" or "01"
GNN_LR = "0001" # or "001" or "01"

architectures = {"GAT", "GCN", "GIN", "SAGE"}
types = {"RBRICS"}
valid_folds = set(map(str, range(5)))

motif_frequency_threshold = {
    "esol": 10, "BBBP": 18, "Lipophilicity": 38, "Mutagenicity": 70,
    "Benzene": 70, "Alkane_Carbonyl": 50, "Fluoride_Carbonyl": 50,
    "hERG": 90, "tox21": 70,
}

folder_pattern = re.compile(
    fr"EXPT-\d+[A-Z]*-([\w_]+)-SEED-\d+-FOLD-(\d+)-(\w+)-EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-([\w\d]+|None)-(RBRICS|None)$"
)

# Store collected data and existing keys
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
training_losses = defaultdict(lambda: defaultdict(list))
validation_losses = defaultdict(lambda: defaultdict(list))
found_keys = set()

# First pass: scan all folders once
for dataset_name, root_dir in root_dirs_dict.items():
    for folder in os.listdir(root_dir):
        match = folder_pattern.match(folder)
        if not match:
            continue
        dset, fold, arch, _, model_type = match.groups()
        if dset != dataset_name or arch not in architectures or model_type not in types or fold not in valid_folds:
            continue

        key = (dataset_name, arch, model_type, fold)

        folder_path = os.path.join(root_dir, folder)
        dfs_all = []
        
        ###########################################################
        # Path to training loss CSV
        training_path = os.path.join(root_dir, folder, "explainer", f"{dataset_name}.csv")
        if not os.path.exists(training_path):
            print(f"Missing explainer file: {training_path}")
            continue
        try:
            df_train = pd.read_csv(training_path)
            if "Train Loss" not in df_train.columns or "Val Loss" not in df_train.columns:
                print(f"'Train Loss' or 'Val Loss' missing in {training_path}")
                continue

            training_losses[dataset_name][arch].append(df_train["Train Loss"].values)
            validation_losses[dataset_name][arch].append(df_train["Val Loss"].values)
            
        except Exception as e:
            print(f"Error reading {training_path}: {e}")
            continue
            
        #########################################################
            
        all_files_valid = True   
        file_paths = [
                os.path.join(folder_path, f"{dataset_name}_explanation_result_with_train.csv"),
                os.path.join(folder_path, f"{dataset_name}_explanation_result_with_validation.csv"),
                os.path.join(folder_path, f"{dataset_name}_explanation_result_with_test.csv")
            ]
        for file in file_paths:
            try:
                df = pd.read_csv(file)
                motif_counts = df["motif"].value_counts()
                df = df[df["motif"].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                df["logit_diff"] = np.abs(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                df["diff_sign"] = np.sign(Sigmoid(df["original_logit"]) - Sigmoid(df["new_logit"]))
                df["sigmoid_bin"] = pd.cut(df["sigmoid_importance"], bins=np.linspace(0, 1, 11), include_lowest=True)
                dfs_all.append(df)
            except Exception as e:
                # print(f"Error reading {file}: {e}")
                all_files_valid = False
        #########################################################
        if all_files_valid:       
            found_keys.add(key)
        # if dataset_name == 'Benzene':
        #     input(found_keys)

        if dfs_all:
            df_all = pd.concat(dfs_all, ignore_index=True)
            plotting_data[arch][dataset_name][model_type].append(df_all)
            print(f"[{dataset_name}] {arch}-{model_type} | Fold {fold} | Samples: {len(df_all)}")

# Merge folds for each configuration
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )

# Missing combinations
missing_data = []
for dataset_name in root_dirs_dict:
    for arch in architectures:
        for model_type in types:
            for fold in valid_folds:
                if (dataset_name, arch, model_type, fold) not in found_keys:
                    missing_data.append((dataset_name, arch, model_type, fold))

# Report
print("\n==== Missing Files Report ====")
if not missing_data:
    print("All expected files were found.")
else:
    for dset, arch, mtype, fold in missing_data:
        print(f"Missing: Dataset={dset}, Architecture={arch}, Type={mtype}, Fold={fold}")



==== Missing Files Report ====
Missing: Dataset=esol, Architecture=SAGE, Type=RBRICS, Fold=2
Missing: Dataset=Benzene, Architecture=SAGE, Type=RBRICS, Fold=4
Missing: Dataset=Alkane_Carbonyl, Architecture=SAGE, Type=RBRICS, Fold=4
Missing: Dataset=Alkane_Carbonyl, Architecture=GIN, Type=RBRICS, Fold=4
Missing: Dataset=Fluoride_Carbonyl, Architecture=SAGE, Type=RBRICS, Fold=4
Missing: Dataset=Fluoride_Carbonyl, Architecture=GIN, Type=RBRICS, Fold=4


In [4]:
found_keys

{('Alkane_Carbonyl', 'GAT', 'RBRICS', '3'),
 ('Alkane_Carbonyl', 'GAT', 'RBRICS', '4'),
 ('Alkane_Carbonyl', 'GCN', 'RBRICS', '2'),
 ('Alkane_Carbonyl', 'GCN', 'RBRICS', '4'),
 ('Alkane_Carbonyl', 'GIN', 'RBRICS', '3'),
 ('Alkane_Carbonyl', 'GIN', 'RBRICS', '4'),
 ('Alkane_Carbonyl', 'SAGE', 'RBRICS', '1'),
 ('BBBP', 'GAT', 'RBRICS', '0'),
 ('BBBP', 'GAT', 'RBRICS', '2'),
 ('BBBP', 'GAT', 'RBRICS', '4'),
 ('BBBP', 'GCN', 'RBRICS', '0'),
 ('BBBP', 'GCN', 'RBRICS', '1'),
 ('BBBP', 'GCN', 'RBRICS', '3'),
 ('BBBP', 'GCN', 'RBRICS', '4'),
 ('BBBP', 'GIN', 'RBRICS', '0'),
 ('BBBP', 'GIN', 'RBRICS', '2'),
 ('BBBP', 'GIN', 'RBRICS', '4'),
 ('BBBP', 'SAGE', 'RBRICS', '0'),
 ('BBBP', 'SAGE', 'RBRICS', '2'),
 ('BBBP', 'SAGE', 'RBRICS', '4'),
 ('Benzene', 'GAT', 'RBRICS', '1'),
 ('Benzene', 'GAT', 'RBRICS', '4'),
 ('Benzene', 'GCN', 'RBRICS', '1'),
 ('Benzene', 'GCN', 'RBRICS', '4'),
 ('Benzene', 'GIN', 'RBRICS', '0'),
 ('Benzene', 'GIN', 'RBRICS', '4'),
 ('Benzene', 'SAGE', 'RBRICS', '3'),
 ('Flu

In [3]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator

arch_colors = {
    "GAT": "tab:blue",
    "GCN": "tab:orange",
    "GIN": "tab:green",
    "SAGE": "tab:red"
}

# Set global matplotlib styles for professional plots
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "lines.linewidth": 2.0,
})

# Ensure output directory exists
output_dir = f"{folder_name}/plots/training_val_loss_plots"
os.makedirs(output_dir, exist_ok=True)

for dataset_name in training_losses:
    for loss_dict, loss_type in zip(
        [training_losses[dataset_name], validation_losses[dataset_name]],
        ["Train Loss", "Validation Loss"]
    ):
        fig, axes = plt.subplots(len(expl_lr_values), len(gnn_lr_values),
                                 figsize=(5 * len(gnn_lr_values), 4 * len(expl_lr_values)),
                                 sharex=True, sharey=False)
        axes = np.atleast_2d(axes)  # Ensure axes is 2D

        # Precompute row min/max values for y-limits
        row_minmax_values = []
        for i, expl_lr in enumerate(expl_lr_values):
            row_min = float('inf')
            row_max = 0
            for arch in architectures:
                for gnn_lr in gnn_lr_values:
                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)
                        row_min = min(row_min, np.min(mean_loss - std_loss))
                        row_max = max(row_max, np.max(mean_loss + std_loss))
            row_minmax_values.append((max(0, row_min), row_max))  # Clamp lower y-limit at 0

        for i, expl_lr in enumerate(expl_lr_values):
            for j, gnn_lr in enumerate(gnn_lr_values):
                ax = axes[i, j]
                active_architectures = []  # Track which architectures were actually plotted

                for arch in architectures:
                    # Skip GIN only for Validation Loss and GNN LR = 0.01
                    if ("validation" in loss_type.lower() and gnn_lr == "01" and arch == "GIN"):
                        continue

                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)

                        epochs = np.arange(min_len)
                        ax.plot(epochs, mean_loss, label=arch, color=arch_colors[arch])
                        ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss,
                                        color=arch_colors[arch], alpha=0.15)
                        active_architectures.append(arch)

                # Set individual y-limits per subplot
                all_values = []
                for arch in active_architectures:
                    if (arch in loss_dict and
                        expl_lr in loss_dict[arch] and
                        gnn_lr in loss_dict[arch][expl_lr]):
                        runs = loss_dict[arch][expl_lr][gnn_lr]
                        min_len = min(len(r) for r in runs)
                        losses = np.array([r[:min_len] for r in runs])
                        mean_loss = losses.mean(axis=0)
                        std_loss = losses.std(axis=0)
                        all_values.append(mean_loss - std_loss)
                        all_values.append(mean_loss + std_loss)

                # Flatten and filter all_values
                flat_values = np.concatenate([v.ravel() for v in all_values if isinstance(v, np.ndarray) and v.size > 0], axis=0)

                ymin = max(0, np.min(flat_values))
                ymax = np.max(flat_values)
                ax.set_ylim(ymin, ymax)
               
                ax.set_yscale('log')
                ax.yaxis.set_major_locator(LogLocator(base=10.0))
                ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1.1, 10) * 0.1, numticks=10))

                if i == len(expl_lr_values) - 1:
                    ax.set_xlabel("Epoch", fontsize=13)

                ax.tick_params(width=1.2)
                ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

                if i == 0:
                    ax.set_title(f"GNN LR = 0.{gnn_lr}", fontsize=14, pad=10)

                if j == 0:
                    ax.annotate(f"Expl LR = 0.{expl_lr}",
                                xy=(-0.35, 0.5),
                                xycoords='axes fraction',
                                ha='center', va='center',
                                rotation=90, fontsize=14)

        
        
        # Add grid-level labels
        fig.text(0.5, 0.04, "Epoch", ha="center", va="center", fontsize=15)
        fig.text(0.04, 0.5, f"{loss_type} Log Scaled", ha="center", va="center", rotation="vertical", fontsize=15)
        fig.text(0.5, 0.97, f"{dataset_name} - {loss_type}", ha="center", va="center", fontsize=16, weight='bold')

        # Build handles only from architectures actually plotted across all subplots
        unique_active_architectures = set()
        for ax_row in axes:
            for ax in ax_row:
                unique_active_architectures.update([line.get_label() for line in ax.get_lines()])

        handles = [Line2D([0], [0], color=arch_colors[arch], label=arch)
                   for arch in unique_active_architectures if arch in arch_colors]

        fig.legend(handles=handles, title="Architecture", loc="center right",
                   bbox_to_anchor=(1.02, 0.5), fontsize=12, title_fontsize=13)
        

        fig.tight_layout(rect=[0.06, 0.06, 0.94, 0.94], pad=2.0)

        # Save figure
        out_file = f"{output_dir}/{dataset_name}_{loss_type.replace(' ', '_').lower()}_heatmap.png"
        fig.savefig(out_file, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"✅ Saved plot: {out_file}")


✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/Mutagenicity_train_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/Mutagenicity_validation_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/hERG_train_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/hERG_validation_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/BBBP_train_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/BBBP_validation_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/esol_train_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss_plots/esol_validation_loss_heatmap.png
✅ Saved plot: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/plots/training_val_loss

In [60]:
arch_colors = {
    "GAT": "tab:blue",
    "GCN": "tab:orange",
    "GIN": "tab:green",
    "SAGE": "tab:red"
}

output_dir = f"{folder_name}/plots/training_val_loss_plots"
os.makedirs(output_dir, exist_ok=True)

for dataset_name in training_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

    for loss_dict, ax, title in zip(
        [training_losses[dataset_name], validation_losses[dataset_name]],
        axes,
        ["Train Loss", "Validation Loss"]
    ):
        for arch in architectures:
            if arch in loss_dict:
                runs = loss_dict[arch]
                min_len = min(len(r) for r in runs)
                losses = np.array([r[:min_len] for r in runs])
                mean_loss = losses.mean(axis=0)
                std_loss = losses.std(axis=0)

                epochs = np.arange(min_len)
                ax.plot(epochs, mean_loss, label=arch, color=arch_colors[arch])
                ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, color=arch_colors[arch], alpha=0.2)

        ax.set_title(f"{title} - {dataset_name}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True)

        # Fixed legend
        handles = [Line2D([0], [0], color=arch_colors[arch], label=arch) for arch in architectures]
        ax.legend(handles=handles, title="Architecture")

    plt.suptitle(f"{dataset_name}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR} Loss Across Folds", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f"{output_dir}/{dataset_name}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}-loss_plot.png", dpi=300)
    plt.close()


## Showing importance to impact. Todo save performance and importance impact correlation to table


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Create base output directory
os.makedirs(f"{folder_name}/plots", exist_ok=True)

# Set seaborn style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

# Constants
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = ["RBRICS"]
n_rows = len(unique_datasets)
n_cols = len(architectures)
palette = "Set2"

for model_type in unique_types:
    for arch in sorted(architectures):
        for dataset in unique_datasets:
            expl_lr_dict = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, {})

            # Iterate over all EXPL_LR keys
            for EXPL_LR in expl_lr_dict.keys():
                gnn_lr_dict = expl_lr_dict.get(EXPL_LR, {})

                # Iterate over all GNN_LR keys
                for GNN_LR in gnn_lr_dict.keys():
                    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 7, n_rows * 5), squeeze=False)
                    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.35, hspace=0.4)

                    for d_idx, dataset in enumerate(unique_datasets):
                        for a_idx, arch in enumerate(sorted(architectures)):
                            ax = axes[d_idx, a_idx]

                            # Access data
                            data = (
                                plotting_data.get(arch, {})
                                             .get(dataset, {})
                                             .get(model_type, {})
                                             .get(EXPL_LR, {})
                                             .get(GNN_LR, pd.DataFrame())
                            )

                            if data.empty:
                                ax.set_visible(False)
                                continue

                            # Filter out UNK motifs
                            data = data[data["motif"] != "UNK"]

                            # Find graph with maximum logit difference per motif
                            idx_max_logit_diff = data.groupby("motif")["logit_diff"].idxmax()
                            graph_max_logit_diff = data.loc[idx_max_logit_diff, ["motif", "graph_str"]].rename(
                                columns={"graph_str": "max_logit_diff_graph"}
                            )

                            # Deduplicate data for motif-graph frequency counting
                            dedup_counts = data.drop_duplicates(subset=["motif", "graph_id"])
                            motif_frequency = dedup_counts.groupby("motif")["graph_id"].nunique().reset_index().rename(
                                columns={"graph_id": "frequency"}
                            )

                            # Compute aggregated statistics
                            motif_stats = data.groupby("motif").agg(
                                avg_sigmoid_importance=("sigmoid_importance", "mean"),
                                sigmoid_bin=("sigmoid_bin", "first"),
                                median_logit_diff=("logit_diff", "median"),
                                mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                                mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                                avg_logit_diff=("logit_diff", "mean"),
                                max_logit_diff=("logit_diff", "max"),
                                duplicate_counts=("motif", "count")
                            ).reset_index()

                            # Merge frequency and max-logit-graph info
                            motif_stats = (
                                motif_stats
                                .merge(motif_frequency, on="motif", how="left")
                                .merge(graph_max_logit_diff, on="motif", how="left")
                            )

                            # Sort and extract top and bottom 10 motifs
                            top_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=False).head(10)
                            bottom_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=True).head(10)

                            # Save to CSV
                            output_csv_dir = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR{EXPL_LR}-GNNLR{GNN_LR}"
                            os.makedirs(output_csv_dir, exist_ok=True)
                            top_10_path = f"{output_csv_dir}/{dataset}_{arch}_top10.csv"
                            bottom_10_path = f"{output_csv_dir}/{dataset}_{arch}_bottom10.csv"
                            top_10_motifs.to_csv(top_10_path, index=False)
                            bottom_10_motifs.to_csv(bottom_10_path, index=False)

                            # Plotting
                            counts = data['sigmoid_bin'].value_counts().sort_index()
                            counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()

                            sns.boxplot(
                                ax=ax,
                                x='sigmoid_bin',
                                y='avg_logit_diff',  # Change to 'median_logit_diff' or 'mode_logit_diff' if needed
                                data=motif_stats,
                                color='tab:blue',
                                width=0.6,
                                linewidth=1.5
                            )

                            if not counts_percentage.empty:
                                ax.bar(
                                    x=np.arange(len(counts_percentage)),
                                    height=-counts_percentage.values,
                                    color='tab:orange',
                                    alpha=0.4,
                                    width=0.5,
                                    align='center'
                                )

                            ax.axhline(0, color='black', linewidth=1, linestyle='--')
                            ax.set_ylim(-0.6, 0.6)
                            ax.set_yticks([-0.5, -0.25, 0, 0.25, 0.5, 1.0])
                            ax.set_yticklabels(['50%', '25%', '0', '0.25', '0.5', '1.0'])

                            ax.set_xlabel('Normalized Motif Importance', fontsize=12)
                            if a_idx == 0:
                                ax.set_ylabel('Freq | Abs Prob Diff', fontsize=12)
                            else:
                                ax.set_ylabel('')
                            
                            ax.tick_params(axis='x', rotation=45)

                    # Row labels (datasets)
                    for d_idx, dataset in enumerate(unique_datasets):
                        pos = axes[d_idx][0].get_position()
                        fig.text(pos.x0 - 0.05, pos.y0 + pos.height / 2, dataset,
                                 ha='right', va='center', rotation=90, fontsize=18, fontweight='bold')

                    # Column labels (architectures)
                    for a_idx, arch in enumerate(sorted(architectures)):
                        pos = axes[0][a_idx].get_position()
                        fig.text(pos.x0 + pos.width / 2, pos.y1 + 0.03, arch,
                                 ha='center', va='bottom', fontsize=18, fontweight='bold')

                    # Save figure
                    output_plot_path = f"{folder_name}/plots/{model_type}-EXPLLR{EXPL_LR}-GNNLR{GNN_LR}_combined_comparison.png"
                    plt.savefig(output_plot_path, bbox_inches='tight')
                    plt.close()

print(f"✅ Saved all plots and CSVs to {folder_name}/plots/")


In [61]:
import matplotlib.pyplot as plt
# Create output directories
os.makedirs(f"{folder_name}/plots", exist_ok=True)
os.makedirs(f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}/", exist_ok=True)

# Set seaborn style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)

# Constants
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = ["RBRICS"]
n_rows = len(unique_datasets)
n_cols = len(architectures)
palette = "Set2"

for model_type in unique_types:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 7, n_rows * 5), squeeze=False)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.35, hspace=0.4)
    
    for d_idx, dataset in enumerate(unique_datasets):
        for a_idx, arch in enumerate(sorted(architectures)):
            ax = axes[d_idx, a_idx]
            data = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, pd.DataFrame())
            # print(arch,dataset,model_type)
            # input(plotting_data[arch][dataset][model_type])

            if data.empty:
                ax.set_visible(False)
                continue
                
            # Step 1: Find the graph with maximum logit difference per motif (using full data)
            idx_max_logit_diff = data.groupby("motif")["logit_diff"].idxmax()
            graph_max_logit_diff = data.loc[idx_max_logit_diff, ["motif", "graph_str"]].rename(columns={"graph_str": "max_logit_diff_graph"})

            # Step 2: Create deduplicated data for motif-graph frequency counting
            dedup_counts = data.drop_duplicates(subset=["motif", "graph_id"])
            motif_frequency = dedup_counts.groupby("motif")["graph_id"].nunique().reset_index().rename(columns={"graph_id": "frequency"})
            
            data = data[data["motif"] != "UNK"]

            # Step 3: Compute all other statistics from full data
            motif_stats = data.groupby("motif").agg(
                avg_sigmoid_importance=("sigmoid_importance", "mean"),
                sigmoid_bin=("sigmoid_bin", "first"),
                median_logit_diff=("logit_diff", "median"),
                mode_sign=("diff_sign", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                mode_logit_diff=("logit_diff", lambda x: pd.Series.mode(x).iloc[0] if not pd.Series.mode(x).empty else np.nan),
                avg_logit_diff=("logit_diff", "mean"),
                max_logit_diff=("logit_diff", "max"),
                duplicate_counts=("motif", "count")
            ).reset_index()

            # Step 4: Merge frequency and max-logit-graph info
            motif_stats = (
                motif_stats
                .merge(motif_frequency, on="motif", how="left")
                .merge(graph_max_logit_diff, on="motif", how="left")
            )

            # Sort and extract top and bottom 10 motifs
            top_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=False).head(10)
            bottom_10_motifs = motif_stats.sort_values(by="avg_sigmoid_importance", ascending=True).head(10)

            # Save to CSV
            top_10_path = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}/{dataset}_{arch}_top10.csv"
            bottom_10_path = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}/{dataset}_{arch}_bottom10.csv"
            top_10_motifs.to_csv(top_10_path, index=False)
            bottom_10_motifs.to_csv(bottom_10_path, index=False)

            # Plotting
            counts = data['sigmoid_bin'].value_counts().sort_index()
            # print(counts)
            counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()

            sns.boxplot(
                ax=ax,
                x='sigmoid_bin',
                y='avg_logit_diff', #Use max median mode
                data=motif_stats,
                color='tab:blue',
                width=0.6,
                linewidth=1.5
            )

            if not counts_percentage.empty:
                ax.bar(
                    x=np.arange(len(counts_percentage)),
                    height=-counts_percentage.values,
                    color='tab:orange',
                    alpha=0.4,
                    width=0.5,
                    align='center'
                )

            ax.axhline(0, color='black', linewidth=1, linestyle='--')
            ax.set_ylim(-0.6, 0.6)
            ax.set_yticks([-0.5, -0.25, 0, 0.25, 0.5, 1.0])
            ax.set_yticklabels(['50%', '25%', '0', '0.25', '0.5', '1.0'])

            ax.set_xlabel('Normalized Motif Importance', fontsize=12)
            if a_idx == 0:
                ax.set_ylabel('Freq | Abs Prob Diff', fontsize=12)
            else:
                ax.set_ylabel('')
            
            ax.tick_params(axis='x', rotation=45)

    # Row labels (datasets)
    for d_idx, dataset in enumerate(unique_datasets):
        pos = axes[d_idx][0].get_position()
        fig.text(pos.x0 - 0.05, pos.y0 + pos.height / 2, dataset,
                 ha='right', va='center', rotation=90, fontsize=18, fontweight='bold')

    # Column labels (architectures)
    for a_idx, arch in enumerate(sorted(architectures)):
        pos = axes[0][a_idx].get_position()
        fig.text(pos.x0 + pos.width / 2, pos.y1 + 0.03, arch,
                 ha='center', va='bottom', fontsize=18, fontweight='bold')

    # Save figure
    plt.savefig(f"{folder_name}/plots/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_combined_comparison.png", bbox_inches='tight')
    plt.close()

print(f"✅ Saved all combined plots and CSVs to {folder_name}/plots/")

✅ Saved all combined plots and CSVs to RBRICS_Minority_MP2_DIM16/plots/


In [6]:
import os
import re
import json
import pandas as pd
from collections import defaultdict

expl_lr_values = ["0001", "001", "01"]
gnn_lr_values = ["0001", "001", "01"]

root_dirs_dict = {
    "Mutagenicity": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "hERG": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "BBBP": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "esol": (f"{folder_name}/MOSE_Reg", f"{folder_name}/Vanilla_Reg"),
    "Lipophilicity": (f"{folder_name}/MOSE_Reg", f"{folder_name}/Vanilla_Reg"),
    "Benzene": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "Alkane_Carbonyl": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "Fluoride_Carbonyl": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
}

architectures = {"GAT", "GCN", "GIN", "SAGE"}

all_rows = []

for EXPL_LR in expl_lr_values:
    for GNN_LR in gnn_lr_values:
        print(f"\n🔍 Processing EXPLLR=0.{EXPL_LR}, GNNLR=0.{GNN_LR}...")
        performance_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

        # Regex for MOSE and Vanilla folders
        folder_pattern = re.compile(
            r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"
            r"(?P<dataset>[\w_]+)-"
            r"SEED-(?P<seed>\d+)-"
            r"FOLD-(?P<fold>\d+)-"
            r"(?P<arch>\w+)-"
            fr"EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-SingleChannel-RBRICS$"
        )

        folder_pattern_vanilla = re.compile(
            r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"
            r"(?P<dataset>[\w_]+)-"
            r"SEED-(?P<seed>\d+)-"
            r"FOLD-(?P<fold>\d+)-"
            r"(?P<arch>\w+)-"
            r"MP2-DIM16-Vanilla-None$"
        )

        for dataset_name, (expl_root, van_root) in root_dirs_dict.items():
            # Index Vanilla folders
            vanilla_lookup = {}
            for vfolder in os.listdir(van_root):
                m = folder_pattern_vanilla.match(vfolder)
                if not m:
                    continue
                key = (m["dataset"], m["seed"], m["fold"], m["arch"])
                vanilla_lookup[key] = vfolder

            # Walk through MOSE folders
            for efolder in os.listdir(expl_root):
                m = folder_pattern.match(efolder)
                if not m:
                    continue

                dataset, seed, fold, arch = (
                    m["dataset"],
                    m["seed"],
                    m["fold"],
                    m["arch"],
                )

                if arch not in architectures:
                    continue

                key = (dataset, seed, fold, arch)
                vfolder = vanilla_lookup.get(key)
                if not vfolder:
                    print(f"⚠️ Vanilla folder missing for {key}")
                    continue

                mose_file = os.path.join(expl_root, efolder, f"{dataset}_classification_result.json")
                vanilla_file = os.path.join(van_root, vfolder, f"{dataset}_classification_result.json")

                files = {"mose": mose_file, "vanilla": vanilla_file}
                missing = [name for name, path in files.items() if not os.path.exists(path)]
                if missing:
                    print(f"⚠️ Missing files for run {efolder}:")
                    for name in missing:
                        print(f"    {name}: {files[name]}")
                    continue

                with open(mose_file) as f:
                    perf_mose = json.load(f)
                with open(vanilla_file) as f:
                    perf_vanilla = json.load(f)

                metric_suffix = "rocauc" if dataset in {"BBBP", "Mutagenicity", "hERG", "Alkane_Carbonyl", "Benzene", "Fluoride_Carbonyl", "tox21"} else "rmse"

                for split in ["train", "validation", "test"]:
                    mose_metric = perf_mose.get(f"Trained_explainations_{split}_{metric_suffix}", None)
                    van_metric = perf_vanilla.get(f"Trained_explainations_{split}_{metric_suffix}", None)
                    performance_data[dataset][arch][f"Mose {split.capitalize()}"].append(mose_metric)
                    performance_data[dataset][arch][f"Vanilla {split.capitalize()}"].append(van_metric)

        # Flatten nested dict
        for dataset, arch_dict in performance_data.items():
            for arch, metrics in arch_dict.items():
                row = {
                    "Dataset": dataset,
                    "Architecture": arch,
                    "EXPL_LR": f"0.{EXPL_LR}",
                    "GNN_LR": f"0.{GNN_LR}"
                }
                for metric_name, values in metrics.items():
                    if values:
                        mean_val = round(sum(values) / len(values), 4)
                        std_val = round(pd.Series(values).std(), 4)
                        row[f"{metric_name} Mean"] = mean_val
                        row[f"{metric_name} Std"] = std_val
                    else:
                        row[f"{metric_name} Mean"] = None
                        row[f"{metric_name} Std"] = None
                all_rows.append(row)

# Combine all rows into one DataFrame
performance_df = pd.DataFrame(all_rows)
performance_df = performance_df.sort_values(by=["Dataset", "Architecture", "EXPL_LR", "GNN_LR"])

# Save big combined CSV
output_csv_path = f"{folder_name}/performance_summary_all_lrs.csv"
os.makedirs(folder_name, exist_ok=True)
performance_df.to_csv(output_csv_path, index=False)
print(f"✅ Combined performance summary with std saved to {output_csv_path}")



🔍 Processing EXPLLR=0.0001, GNNLR=0.0001...
⚠️ Vanilla folder missing for ('Fluoride_Carbonyl', '0', '3', 'GCN')
⚠️ Vanilla folder missing for ('hERG', '0', '3', 'GAT')
⚠️ Vanilla folder missing for ('Fluoride_Carbonyl', '0', '4', 'GAT')
⚠️ Vanilla folder missing for ('hERG', '0', '2', 'GIN')
⚠️ Vanilla folder missing for ('Alkane_Carbonyl', '0', '4', 'GIN')
⚠️ Vanilla folder missing for ('Benzene', '0', '4', 'GAT')
⚠️ Vanilla folder missing for ('Benzene', '0', '2', 'GIN')
⚠️ Vanilla folder missing for ('hERG', '0', '4', 'GIN')
⚠️ Vanilla folder missing for ('hERG', '0', '3', 'GIN')
⚠️ Vanilla folder missing for ('Fluoride_Carbonyl', '0', '2', 'SAGE')
⚠️ Vanilla folder missing for ('Alkane_Carbonyl', '0', '3', 'GAT')
⚠️ Vanilla folder missing for ('Alkane_Carbonyl', '0', '3', 'SAGE')
⚠️ Missing files for run EXPT-14BC-Benzene-SEED-0-FOLD-1-GCN-EXPLLR0.0001-GNNLR0.0001-SingleChannel-RBRICS:
    mose: ../RBRICS_MINORITY_MP2_DIM16_CORRECTED/MOSE_BC/EXPT-14BC-Benzene-SEED-0-FOLD-1-GCN-EX

In [62]:
import json
root_dirs_dict = {
    "Mutagenicity": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "hERG": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "BBBP": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "esol": (f"{folder_name}/MOSE_Reg", f"{folder_name}/Vanilla_Reg"),
    "Lipophilicity": (f"{folder_name}/MOSE_Reg", f"{folder_name}/Vanilla_Reg"),
    "Benzene": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "Alkane_Carbonyl": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
    "Fluoride_Carbonyl": (f"{folder_name}/MOSE_BC", f"{folder_name}/Vanilla_BC"),
}

architectures = {"GAT", "GCN", "GIN", "SAGE"}
types = {"RBRICS", "None"}

# Regex to match MOSE experiment folders with fixed suffix "EXPLLR0.01-SingleChannel-RBRICS"
folder_pattern = re.compile(
    r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"             # EXPT ID
    r"(?P<dataset>[\w_]+)-"                      # Dataset name
    r"SEED-(?P<seed>\d+)-"                       # Seed
    r"FOLD-(?P<fold>\d+)-"                       # Fold
    r"(?P<arch>\w+)-"                            # Architecture
    fr"EXPLLR0\.{EXPL_LR}-GNNLR0\.{GNN_LR}-SingleChannel-RBRICS$"         # Fixed suffix
)

# Regex to match Vanilla experiment folders with fixed suffix "MP3-DIM32-Vanilla-None"
folder_pattern_vanilla = re.compile(
    r"^EXPT-(?P<expt_id>\d+[A-Z]*)-"             # EXPT ID
    r"(?P<dataset>[\w_]+)-"                      # Dataset name
    r"SEED-(?P<seed>\d+)-"                       # Seed
    r"FOLD-(?P<fold>\d+)-"                       # Fold
    r"(?P<arch>\w+)-"                            # Architecture
    r"MP2-DIM16-Vanilla-None$"                   # Fixed suffix
)

# Store performance data
performance_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Iterate over datasets
for dataset_name, (expl_root, van_root) in root_dirs_dict.items():
    
    # Index vanilla folders
    vanilla_lookup = {}
    for vfolder in os.listdir(van_root):
        m = folder_pattern_vanilla.match(vfolder)
        
        if not m:
            continue
        key = (m["dataset"], m["seed"], m["fold"], m["arch"])
        vanilla_lookup[key] = vfolder

    # Walk through explainer folders
    for efolder in os.listdir(expl_root):
        m = folder_pattern.match(efolder)
        if not m:
            continue

        dataset, seed, fold, arch = (
            m["dataset"],
            m["seed"],
            m["fold"],
            m["arch"],
        )

        if arch not in architectures:
            continue

        key = (dataset, seed, fold, arch)
        vfolder = vanilla_lookup.get(key)
        if not vfolder:
            continue

        mose_file = os.path.join(expl_root, efolder, f"{dataset}_classification_result.json")
        vanilla_file = os.path.join(van_root, vfolder, f"{dataset}_classification_result.json")

        files = {
            "mose": mose_file,
            "vanilla": vanilla_file,
        }

        missing = [name for name, path in files.items() if not os.path.exists(path)]
        if missing:
            print(f"⚠️ Missing files for run {efolder}:")
            for name in missing:
                print(f"    {name}: {files[name]}")
            print("Skipping this run.\n")
            continue

        with open(mose_file) as f:
            perf_mose = json.load(f)
        with open(vanilla_file) as f:
            perf_vanilla = json.load(f)

        metric_suffix = "rocauc" if dataset in {"BBBP", "Mutagenicity", "hERG","Alkane_Carbonyl","Benzene","Fluoride_Carbonyl","tox21"} else "rmse"

        for split in ["train", "validation", "test"]:
            mose_metric = perf_mose.get(f"Trained_explainations_{split}_{metric_suffix}", 0)
            van_metric = perf_vanilla.get(f"Trained_explainations_{split}_{metric_suffix}", 0)
            performance_data[dataset][arch][f"Mose {split.capitalize()} Metric"].append(mose_metric)
            performance_data[dataset][arch][f"Vanilla {split.capitalize()} Metric"].append(van_metric)

# Flatten nested dictionary into table
rows = []
for dataset, arch_dict in performance_data.items():
    for arch, metric_dict in arch_dict.items():
        
        row = {
            "Dataset": dataset,
            "Architecture": arch,
        }
        for metric_name, values in metric_dict.items():
            row[metric_name] = round(sum(values) / len(values), 4) if values else None
        rows.append(row)

performance_df = pd.DataFrame(rows)
performance_df = performance_df.sort_values(by=["Dataset", "Architecture"])

# Save the output CSV
output_csv_path = f"{folder_name}/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_performance_comparison.csv"
os.makedirs(folder_name, exist_ok=True)
performance_df.to_csv(output_csv_path, index=False)
print(f"✅ Performance summary saved to {output_csv_path}")

⚠️ Missing files for run EXPT-14R-Lipophilicity-SEED-0-FOLD-1-GAT-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS:
    mose: RBRICS_Minority_MP2_DIM16/MOSE_Reg/EXPT-14R-Lipophilicity-SEED-0-FOLD-1-GAT-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS/Lipophilicity_classification_result.json
Skipping this run.

⚠️ Missing files for run EXPT-14R-Lipophilicity-SEED-0-FOLD-0-GAT-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS:
    mose: RBRICS_Minority_MP2_DIM16/MOSE_Reg/EXPT-14R-Lipophilicity-SEED-0-FOLD-0-GAT-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS/Lipophilicity_classification_result.json
Skipping this run.

⚠️ Missing files for run EXPT-14R-Lipophilicity-SEED-0-FOLD-1-SAGE-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS:
    mose: RBRICS_Minority_MP2_DIM16/MOSE_Reg/EXPT-14R-Lipophilicity-SEED-0-FOLD-1-SAGE-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS/Lipophilicity_classification_result.json
Skipping this run.

⚠️ Missing files for run EXPT-14R-esol-SEED-0-FOLD-0-SAGE-EXPLLR0.01-GNNLR0.01-SingleChannel-RBRICS:
   

In [ ]:
import pandas as pd
import os
import glob

# Directory containing CSV files
csv_folder = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}"  # Change this path if needed

# File pattern
csv_files = glob.glob(os.path.join(csv_folder, "*_*_top10.csv")) + \
            glob.glob(os.path.join(csv_folder, "*_*_bottom10.csv"))

# Columns to retain
columns_to_keep = [
    "motif", "avg_sigmoid_importance", "median_logit_diff", "mode_sign",
    "mode_logit_diff", "avg_logit_diff", "max_logit_diff", "duplicate_counts",
    "frequency", "max_logit_diff_graph"
]

all_data = []

for file in csv_files:
    try:
        filename = os.path.basename(file).replace(".csv", "")
        parts = filename.split("_")

        # Parse dataset name, architecture, and rank type
        if parts[-1] in {"top10", "bottom10"}:
            label = parts[-1]
            architecture = parts[-2]
            dataset = "_".join(parts[:-2])
        else:
            print(f"Skipping invalid filename format: {filename}")
            continue

        df = pd.read_csv(file)

        # Ensure required columns are present
        if not set(columns_to_keep).issubset(df.columns):
            print(f"Skipping {filename}: missing required columns")
            continue

        # Select and round float columns
        df = df[columns_to_keep]
        float_cols = df.select_dtypes(include=['float']).columns
        df[float_cols] = df[float_cols].round(3)

        # Add metadata
        df["dataset"] = dataset
        df["architecture"] = architecture
        df["rank_type"] = label

        all_data.append(df)
    except Exception as e:
        print(f"Error processing {file}: {e}")

# Combine all data
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df.to_csv(f"{csv_folder}/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_combined_motif_stats.csv", index=False)
    print("Saved combined_motif_stats.csv")
else:
    print("No valid data to combine.")

IndentationError: expected an indented block after 'for' statement on line 16 (2103916973.py, line 19)

In [ ]:
input()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from glob import glob

def plot_motif_bias(csv_file, combined_csv, min_support=1, save_dir=None):
    # Load per-fold data
    df = pd.read_csv(csv_file)
    df['Motif Length'] = pd.to_numeric(df.get('Motif Length', 0), errors='coerce')
    df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
    df = df.drop_duplicates(subset=['Graph String', 'Motif String'])

    # Rename for merge
    df = df.rename(columns={'Motif String': 'motif'})
    
    # Determine dataset name
    filename = os.path.basename(csv_file).replace(".csv", "")
    parts = filename.rsplit("_", 2)
    dataset_name = parts[0].lower() 

    # Load combined motif stats (with architecture + rank_type)
    combined_df = pd.read_csv(combined_csv)
    filtered_combined = combined_df[
        (combined_df['dataset'].str.lower() == dataset_name)
    ].copy()

    motif_meta = filtered_combined[['motif', 'rank_type', 'architecture']].drop_duplicates()

    # motif_meta = combined_df[['motif', 'architecture', 'rank_type', 'dataset']].drop_duplicates()
    # motif_meta = motif_meta[
    #         (combined_df['dataset'].str == dataset_name)
    #     ].copy()
    # input(f"{filtered_combined['dataset'].str.lower()} {dataset_name}")
    # input(motif_meta)
    df = df.merge(motif_meta, on='motif', how='left')
    df['rank_type'] = df['rank_type'].apply(lambda x: x if x in ['top10', 'bottom10'] else 'other')
    df['architecture'] = df['architecture'].fillna('default')

     

    is_regression = dataset_name in ['lipophilicity', 'esol']
    if not is_regression:
        df = df[df['Label'].isin([0, 1])]

    # Define color map for rank_type
    rank_colors = {
        'top10': 'tab:blue',
        'bottom10': 'tab:red',
        'other': 'tab:gray'
    }

    # Loop per architecture
    for arch in df['architecture'].unique():
        arch_df = df[(df['architecture'] == arch) | (df['architecture'] == 'default')].copy()
        # arch_df['rank_type'] = arch_df['rank_type'].fillna('other')
        # input(arch_df['rank_type'].unique())

        # Group motif stats
        counts = arch_df.groupby(['motif', 'rank_type']).size().reset_index(name='count')
        motif_total = arch_df.groupby('motif').size().reset_index(name='total')
        class_ratio = arch_df.groupby('motif')['Label'].mean().reset_index(name='class1_ratio')
        merged = counts.merge(motif_total, on='motif').merge(class_ratio, on='motif')

        # Filter
        merged = merged[merged['total'] >= min_support]

        # Plot
        fig, ax = plt.subplots(figsize=(8, 6))
        plotted_labels = set()
        for _, row in merged.iterrows():
            rt = row.get('rank_type', 'unknown')
            color = rank_colors.get(rt, 'black')
            label = rt if rt not in plotted_labels else None

            if pd.isna(rt) or rt == 'unknown':
                ax.scatter(
                    row['total'],
                    row['class1_ratio'],
                    facecolors='white',
                    edgecolors='black',
                    linewidth=0.8,
                    linestyle='--',
                    s=60,
                    label='unranked' if 'unranked' not in plotted_labels else None
                )
                plotted_labels.add('unranked')
            elif rt == 'bottom10':
                ax.scatter(
                    row['total'],
                    row['class1_ratio'],
                    facecolors='none',
                    edgecolors=color,
                    linewidth=1.2,
                    s=60,
                    label=label
                )
                plotted_labels.add(rt)
            else:
                ax.scatter(
                    row['total'],
                    row['class1_ratio'],
                    facecolors=color,
                    edgecolors='k',
                    linewidth=0.5,
                    s=60,
                    label=label
                )
                plotted_labels.add(rt)

        # Annotate high-frequency motifs
        ax.figure.canvas.draw()
        xlim = ax.get_xlim()
        freq_thresh = 0.4 * xlim[1]
        for _, row in merged.iterrows():
            rt = row['rank_type']
            color = rank_colors.get(rt, 'tab:gray')
            face = 'none' if rt == 'bottom10' else color
            label = rt if rt not in plotted_labels else None
            ax.scatter(
                row['total'],
                row['class1_ratio'],
                facecolors=face,
                edgecolors=color,
                linewidth=0.8,
                s=50,
                label=label
            )
            if label:
                plotted_labels.add(rt)

        ax.axhline(0.5, color='gray', linestyle='--')
        ax.set_xlabel("Motif Frequency")
        ax.set_ylabel("Class 1 Ratio" if not is_regression else "Mean Label")
        ax.set_title(f"{dataset_name.upper()} | {arch} | Fold 0", fontsize=11)

        ax.legend(title="Rank Type", fontsize=8, title_fontsize=9)
        plt.tight_layout()

        if save_dir:
            out_path = os.path.join(save_dir, f"{dataset_name}_{arch}_motif_bias_fold0.png")
            fig.savefig(out_path, dpi=300)
            print(f"✅ Saved: {out_path}")
            plt.close(fig)
        else:
            plt.show()

# === Main execution ===
plot_folder = f"{folder_name}/plots/motif_csvs/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}"
csv_folder = f"{folder_name}/csv_exports"
csv_files = sorted(glob(os.path.join(csv_folder, "*_0_RBRICS.csv")))
combined_motif_stats = f"{plot_folder}/{model_type}-EXPLLR0.{EXPL_LR}-GNNLR0.{GNN_LR}_combined_motif_stats.csv"
plot_dir = f"{plot_folder}/plots"
os.makedirs(plot_dir, exist_ok=True)

for f in csv_files:
    plot_motif_bias(f, combined_motif_stats, save_dir=plot_dir)

In [ ]:
input()

In [13]:
'''
No vanilla
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "hERG":["ent_reg"],
                  "hERG_2":["ent_reg"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type} (\d)
folder_pattern = re.compile(
    r"EXPT-ENTBC+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-(\d*\.?\d+|\d+\.?\d*)+-(MGSSL|RBRICS)"
)

# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                fold,architecture, ent, model_type = match.groups()
                # input(match.groups())
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_paths = [
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoc_train.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoc_validation.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoc_test.csv")
                    ]
                    dfs_correct = []
                    for file in file_paths:
                        try:
                            
                            df = pd.read_csv(file)
                        except Exception as e:
                            print(f"Error reading {file}: {e}")
                            continue

                        
                        dfs_correct.append(df)
                    
                    # Only process further if we have at least one correct sample dataframe.
                    if dfs_correct:
                        df_correct = pd.concat(dfs_correct, ignore_index=True)
                    else:
                        continue
                    
                    # --- MOTIF FREQUENCY FILTERING ---
                    # Count the occurrence of each motif in the correct examples.
                    # It is assumed that there is a column named 'motif_id' in the data.
                    motif_counts = df_correct['motif'].value_counts()
                    # Filter to include only rows where the motif frequency is greater than the threshold.
                    df_correct = df_correct[df_correct['motif'].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                    # Create bins for sigmoid_importance (10 bins)
                    bin_edges = np.linspace(0, 1, 11)
                    # --- CALCULATE ABSOLUTE LOGIT DIFFERENCE ---
                    
                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_correct['logit_diff'] = np.abs(Sigmoid(df_correct['original_logit']) - Sigmoid(df_correct['new_logit']))
                    df_correct['sigmoid_bin'] = pd.cut(df_correct['sigmoid_importance'],
                                                           bins=bin_edges, include_lowest=True)
                    
                    
                    df_combined = df_correct
                    

                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_combined)

# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )

# ================================
# PLOTTING: ONE PDF, EACH PAGE IS ONE ARCHITECTURE
# ================================
# Here the rows of the grid are datasets and the columns are unique types.
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = types  # e.g., ["RBRICS", "MGSSL"]

# Determine grid dimensions:
n_datasets = len(unique_datasets)
n_arch = len(architectures)
n_cols_total = n_datasets * 2  # Two sub-columns per dataset, one per type.
n_rows_total = n_arch  # Rows correspond to architectures.

# Initialize legend handles & labels
legend_handles, legend_labels = None, None

for model_type in unique_types:
    # Determine grid dimensions: datasets as rows, architectures as columns.
    n_rows = len(unique_datasets)
    n_cols = len(architectures)
    
    # Create a figure with appropriate size.
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 8, n_rows * 6), squeeze=True)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.25, hspace=0.35)
    
    legend_handles, legend_labels = None, None  # To capture legend once
    
    # Populate each subplot.
    for d_idx, dataset in enumerate(unique_datasets):
        for a_idx, arch in enumerate(architectures):
            ax = axes[d_idx, a_idx]
            # Retrieve data for this combination.
            data = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, pd.DataFrame())
            
            if not data.empty:
                # Plotting code (similar to original).
                counts = data['sigmoid_bin'].value_counts().sort_index()
                counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()
                # Calculate average for each motif within each sigmoid_bin
                # pdb.set_trace()
                motif_avg = data.groupby(["motif", "sigmoid_bin"])[["logit_diff"]].mean().reset_index()

                # Melt the dataframe for easier plotting
                data_melted = motif_avg.melt(
                    id_vars=["motif", "sigmoid_bin"], 
                    value_vars=["logit_diff"], 
                    var_name="metric", 
                    value_name="value"
                )

                # data_melted = pd.melt(
                #     data, 
                #     id_vars=['sigmoid_bin'], 
                #     value_vars=['logit_diff_vanilla', 'logit_diff'], 
                #     var_name='metric', 
                #     value_name='value'
                # )
                
                sns.boxplot(
                    ax=ax,
                    x='sigmoid_bin',
                    y='value',
                    hue='metric',
                    data=data_melted,
                    width=0.6,
                    dodge=True,
                    linewidth=1.5,
                )
                
                # Capture legend handles once.
                if legend_handles is None:
                    legend_handles, legend_labels = ax.get_legend_handles_labels()
                ax.get_legend().remove()
                
                # Add histogram for percentage counts.
                if not counts_percentage.empty:
                    ax.bar(
                        x=np.arange(len(counts_percentage)),
                        height=-counts_percentage.values,
                        color='tab:orange',
                        alpha=0.5,
                        width=0.4,
                        align='center'
                    )
                
                # Configure axes.
                ax.set_ylim(-0.6)
                ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
                ax.set_yticks([-0.5, -0.25, 0, 0.5])
                ax.set_yticklabels(['50', '25', '0', '0.5'])
                ax.set_xlabel('Normalized Motif Importance', fontsize=14, fontweight='bold')
                ax.tick_params(axis='x', rotation=45)
                
                # Set y-label for the first column.
                if a_idx == 0:
                    ax.set_ylabel(f' % Count     |     Abs Prob Diff', fontweight='bold', fontsize=12)
                else:
                    ax.set_ylabel('')
            else:
                ax.set_visible(False)
    
    # Add dataset labels on the left.
    for d_idx, dataset in enumerate(unique_datasets):
        pos = axes[d_idx, 0].get_position()
        fig.text(pos.x0 - 0.05, pos.y0 + pos.height/2, dataset, 
                 ha='right', va='center', rotation=90, fontsize=22, fontweight='bold')
    
    # Add architecture labels on top.
    for a_idx, arch in enumerate(["GAT", "GCN", "GIN"]):
        pos = axes[0, a_idx].get_position()
        fig.text(pos.x0 + pos.width/2, pos.y1 + 0.02, arch, 
                 ha='center', va='bottom', fontsize=22, fontweight='bold')
    
    # Add global legend.
    if legend_handles and legend_labels:
        fig.legend(
            legend_handles, 
            ['Vanilla', 'MOSE'], 
            title='Model', 
            loc='upper right', 
            bbox_to_anchor=(0.98, 0.93),  # Moves the legend to the top-right
            ncol=2,  # Keep items in a single row
            fontsize=14, 
            title_fontsize=16, 
            frameon=True, 
            edgecolor='black'
        )
    
    # Save the figure.
    plt.savefig(f'{model_type}_ent.png', bbox_inches='tight')
    plt.close()
print("Saved combined plots as combined_plots.png")


Error reading ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_train.csv: [Errno 2] No such file or directory: 'ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_train.csv'
Error reading ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_validation.csv: [Errno 2] No such file or directory: 'ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_validation.csv'
Error reading ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_test.csv: [Errno 2] No such file or directory: 'ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-4-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_test.csv'
Error reading ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-3-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withposthoc_train.csv: [Errno 2] No such file or directory: 'ent_reg/EXPT-ENTBC-hERG-SEED-0-FOLD-3-GCNConv-0.0-RBRICS/hERG_vanilla_impact_withpos

In [ ]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "esol":["0205_ALL/1225esol", "0205_ALL/Vanilla_Reg"],
                  "tox21":["0205_ALL/0205Tox",
                             "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type}
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold, architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold, architecture, model_type = match.groups()
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_paths = [
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_train.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_validation.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_test.csv")
                    ]
                    dfs_correct = []
                    dfs_incorrect = []
                    for file in file_paths:
                        try:
                            
                            df = pd.read_csv(file)
                            if 'original_logit_vanilla' not in df.columns:
                                print(file_paths)
                                continue
                            if dataset_name != "tox21":
                                df.rename(columns={'original_logit_class_0': 'original_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_0': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_vanilla': 'new_logit_vanilla'}, inplace=True)
                                df.rename(columns={'sigmoid_importance_for_class_0': 'sigmoid_importance'}, inplace=True)
                        except Exception as e:
                            print(f"Error reading {file}: {e}")
                            continue

                        
                        
                        dfs_correct.append(df)
                    
                    if dfs_correct:
                        df_correct = pd.concat(dfs_correct, ignore_index=True)
                    else:
                        continue
                    
                    
                    # --- MOTIF FREQUENCY FILTERING ---
                    # Count the occurrence of each motif in the correct examples.
                    # It is assumed that there is a column named 'motif_id' in the data.
                    motif_counts = df_correct['motif'].value_counts()
                    # Filter to include only rows where the motif frequency is greater than the threshold.
                    df_correct = df_correct[df_correct['motif'].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                    # Create bins for sigmoid_importance (10 bins)
                    bin_edges = np.linspace(0, 1, 11)
                    # --- CALCULATE ABSOLUTE LOGIT DIFFERENCE ---
                    
                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_correct['logit_diff'] = np.abs(Sigmoid(df_correct['original_logit']) - Sigmoid(df_correct['new_logit']))
                    df_correct['logit_diff_vanilla'] = np.abs(Sigmoid(df_correct['original_logit_vanilla']) - Sigmoid(df_correct['new_logit_vanilla']))
                    df_correct['sigmoid_imp'] = df_correct['sigmoid_importance']
                    df_correct['class_label'] = df_correct['class_label']
                    
                    # Tag the data (if you want to keep track in the DataFrame)
                    df_correct = df_correct.assign(Correct='Correct')
                    
                    df_combined = df_correct
                    
                    
                    # Update the counts after filtering.
                    num_correct = len(df_correct)
                    print(f"Dataset: {dataset_name}, Architecture: {architecture}, Fold: {fold}, Type: {model_type}")
                    

                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_combined)

# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )

# ================================
# PLOTTING: ONE PDF, EACH PAGE IS ONE ARCHITECTURE
# ================================
# Here the rows of the grid are datasets and the columns are unique types.
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = types  # e.g., ["RBRICS", "MGSSL"]

# Determine grid dimensions:
n_datasets = len(unique_datasets)
n_arch = len(architectures)
n_cols_total = n_datasets * 2  # Two sub-columns per dataset, one per type.
n_rows_total = n_arch  # Rows correspond to architectures.


# Initialize legend handles & labels
legend_handles, legend_labels = None, None

# Adjust subplot parameters to reduce extra white space.
# fig.subplots_adjust(left=0.15, right=0.95, top=0.95, bottom=0.08, wspace=0.25, hspace=0.35)

for model_type in unique_types:
    # Determine grid dimensions: datasets as rows, architectures as columns.
    n_rows = len(unique_datasets)
    n_cols = len(architectures)
    
    # Create a figure with appropriate size.
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 8, n_rows * 6), squeeze=True)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.25, hspace=0.35)
    
    legend_handles, legend_labels = None, None  # To capture legend once
    
    # Populate each subplot.
    for d_idx, dataset in enumerate(unique_datasets):
        for a_idx, arch in enumerate(architectures):
            ax = axes[d_idx, a_idx]
            # Retrieve data for this combination.
            data = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, pd.DataFrame())
            
            if not data.empty:
                # Step 1: Count occurrences of each motif
                motif_counts = data['motif'].value_counts().reset_index()
                motif_counts.columns = ['motif', 'count']

                # Step 2: Calculate total number of motif occurrences
                total_count = motif_counts['count'].max()

                # Step 3: Normalize the counts to get frequencies
                motif_counts['f_i'] = motif_counts['count'] / total_count

                # Merge global frequency into the data
                data_with_f = data.merge(motif_counts[['motif', 'f_i']], on='motif', how='left')


                # Calculate averages for each motif within each sigmoid_bin, including m_i and f_i
                motif_avg = data_with_f.groupby(["motif"]).agg({
                    'logit_diff_vanilla': 'mean',
                    'logit_diff': 'mean',
                    'sigmoid_imp':'first',
                    'f_i': 'first'  # f_i is constant per motif
                }).reset_index()
                # motif_avg=data_with_f

                # Calculate S_i and S_i_vanilla
                epsilon = 1e-8
                motif_avg['S_i'] = - motif_avg['f_i'] * np.log(motif_avg['logit_diff'] + epsilon) - (1 - motif_avg['f_i']) * np.log(1 - motif_avg['logit_diff'] + epsilon)
                motif_avg['S_i_vanilla'] = - motif_avg['f_i'] * np.log(motif_avg['logit_diff_vanilla'] + epsilon) - (1 - motif_avg['f_i']) * np.log(1 - motif_avg['logit_diff_vanilla'] + epsilon)
                motif_avg['S_diff'] = motif_avg['S_i'] - motif_avg['S_i_vanilla']
                
                motif_avg = motif_avg[motif_avg['sigmoid_imp'] > 0.7]


                # Melt the dataframe for easier plotting
                data_melted = motif_avg.melt(
                    id_vars=["motif", "f_i","sigmoid_imp"],  # Include f_i and motif as identifiers
                    value_vars=["logit_diff","logit_diff_vanilla"],#["S_i", "S_i_vanilla"],  # Melt S_i and S_i_vanilla
                    var_name="metric", 
                    value_name="Logit Diff"
                )

                # Define a color palette for the metrics
                palette = {"logit_diff": "#d62728", "logit_diff_vanilla": "#1f77b4"}

                sns.scatterplot(
                    data=data_melted,
                    x='f_i',
                    y='Logit Diff',
                    hue='metric',
                    palette=palette,
                    ax=ax,
                    alpha=0.4
                )

                
                # Capture legend handles once.
                if legend_handles is None:
                    legend_handles, legend_labels = ax.get_legend_handles_labels()
                # ax.get_legend().remove()
                print(data_melted[data_melted['Logit Diff']>0.3])
                
                ax.set_xlim(0, 1)
                
                # Annotate points with 'motif' where f_i > 0.8
                for _, row in data_melted[data_melted['Logit Diff'] > 0.3].iterrows():
                    ax.annotate(
                        row['motif'],
                        (row['f_i'], row['Logit Diff']),  # Position of the point
                        textcoords='offset points',       # Offset text position by points
                        xytext=(5, 5),                    # Offset from the point (x, y)
                        ha='center',                      # Horizontal alignment
                        fontsize=8                        # Adjust font size as needed
                    )
                
                
    
    # Add dataset labels on the left.
    for d_idx, dataset in enumerate(unique_datasets):
        pos = axes[d_idx, 0].get_position()
        fig.text(pos.x0 - 0.05, pos.y0 + pos.height/2, dataset, 
                 ha='right', va='center', rotation=90, fontsize=22, fontweight='bold')
    
    # Add architecture labels on top.
    for a_idx, arch in enumerate(["GAT", "GCN", "GIN"]):
        pos = axes[0, a_idx].get_position()
        fig.text(pos.x0 + pos.width/2, pos.y1 + 0.02, arch, 
                 ha='center', va='bottom', fontsize=22, fontweight='bold')
    
    # Add global legend.
    if legend_handles and legend_labels:
        fig.legend(
            legend_handles, 
            ['MOSE', 'Vanilla'], 
            title='Model', 
            loc='upper right', 
            bbox_to_anchor=(0.98, 0.93),  # Moves the legend to the top-right
            ncol=2,  # Keep items in a single row
            fontsize=14, 
            title_fontsize=16, 
            frameon=True, 
            edgecolor='black'
        )
    
    # Save the figure.
    plt.savefig(f'{model_type}_freq.png', bbox_inches='tight')
    plt.close()
print("Saved combined plots as combined_plots.png")


In [ ]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "esol":["0205_ALL/1225esol", "0205_ALL/Vanilla_Reg"],
                  "tox21":["0205_ALL/0205Tox",
                             "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type}
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)

# Collect all graph IDs per dataset during initial processing
dataset_graph_ids = defaultdict(set)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold, architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold, architecture, model_type = match.groups()
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_paths = [
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_train.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_validation.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_test.csv")
                    ]
                    dfs_correct = []
                    dfs_incorrect = []
                    for file in file_paths:
                        try:
                            
                            df = pd.read_csv(file)
                            dataset_graph_ids[dataset_name].update(df['graph_id'].unique())
                            if 'original_logit_vanilla' not in df.columns:
                                print(file_paths)
                                continue
                            if dataset_name != "tox21":
                                df.rename(columns={'original_logit_class_0': 'original_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_0': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class': 'new_logit'}, inplace=True)
                                df.rename(columns={'new_logit_class_vanilla': 'new_logit_vanilla'}, inplace=True)
                                df.rename(columns={'sigmoid_importance_for_class_0': 'sigmoid_importance'}, inplace=True)
                        except Exception as e:
                            print(f"Error reading {file}: {e}")
                            continue

                        
                        
                        dfs_correct.append(df)
                    
                    if dfs_correct:
                        df_correct = pd.concat(dfs_correct, ignore_index=True)
                    else:
                        continue
                    
                    
                    # --- MOTIF FREQUENCY FILTERING ---
                    # Count the occurrence of each motif in the correct examples.
                    # It is assumed that there is a column named 'motif_id' in the data.
                    motif_counts = df_correct['motif'].value_counts()
                    # Filter to include only rows where the motif frequency is greater than the threshold.
                    df_correct = df_correct[df_correct['motif'].map(motif_counts) > motif_frequency_threshold[dataset_name]]
                    # Create bins for sigmoid_importance (10 bins)
                    bin_edges = np.linspace(0, 1, 11)
                    # --- CALCULATE ABSOLUTE LOGIT DIFFERENCE ---
                    
                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_correct['logit_diff'] = np.abs(Sigmoid(df_correct['original_logit']) - Sigmoid(df_correct['new_logit']))
                    df_correct['logit_diff_vanilla'] = np.abs(Sigmoid(df_correct['original_logit_vanilla']) - Sigmoid(df_correct['new_logit_vanilla']))
                    df_correct['sigmoid_imp'] = df_correct['sigmoid_importance']
                    df_correct['class_label'] = df_correct['class_label']
                    
                    # Tag the data (if you want to keep track in the DataFrame)
                    df_correct = df_correct.assign(Correct='Correct')
                    
                    df_combined = df_correct
                    
                    
                    # Update the counts after filtering.
                    num_correct = len(df_correct)
                    print(f"Dataset: {dataset_name}, Architecture: {architecture}, Fold: {fold}, Type: {model_type}")
                    

                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_combined)
# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )
# ================================
# STRUCTURED OUTPUT FOR HIGH-IMPACT MOTIFS
# ================================
# Create a directory to store the structured output
structured_output_dir = "high_impact_motifs_structured"
os.makedirs(structured_output_dir, exist_ok=True)

# Enhanced structured output generation
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            data = plotting_data[arch][dataset][model_type]
            
            if not data.empty:
                # Initial processing
                motif_counts = data['motif'].value_counts().reset_index()
                motif_counts.columns = ['motif', 'count']
                total_count = motif_counts['count'].sum()
                motif_counts['f_i'] = motif_counts['count'] / total_count
                
                data_with_f = data.merge(motif_counts[['motif', 'f_i']], on='motif', how='left')
                
                motif_avg = data_with_f.groupby(["motif"]).agg({
                    'logit_diff_vanilla': 'mean',
                    'logit_diff': 'mean',
                    'sigmoid_imp': 'first',
                    'f_i': 'first',
                    'graph_id': lambda x: list(x.unique())
                }).reset_index()

                # High-impact filter with lower threshold
                high_impact = motif_avg[
                    (motif_avg['logit_diff'] > 0.2) &  # Lowered threshold
                    (motif_avg['sigmoid_imp'] > 0.5)
                ]
                
                # 1. Ensure graph coverage
                all_graphs = dataset_graph_ids[dataset]
                covered_graphs = set()
                structured_data = []
                
                # Add high-impact motifs
                for _, row in high_impact.iterrows():
                    graphs = sorted(row['graph_id'])
                    structured_data.append({
                        'dataset': dataset,
                        'architecture': arch,
                        'model_type': model_type,
                        'motif': row['motif'],
                        'avg_logit_diff': row['logit_diff'],
                        'graph_ids': graphs
                    })
                    covered_graphs.update(graphs)
                
                # 2. Add missing graphs with their best motifs
                missing_graphs = all_graphs - covered_graphs
                for graph_id in missing_graphs:
                    graph_data = data[data['graph_id'] == graph_id]
                    if not graph_data.empty:
                        best_motif = graph_data.groupby('motif')['logit_diff'].mean().idxmax()
                        structured_data.append({
                            'dataset': dataset,
                            'architecture': arch,
                            'model_type': model_type,
                            'motif': best_motif,
                            'avg_logit_diff': graph_data[graph_data['motif'] == best_motif]['logit_diff'].mean(),
                            'graph_ids': [graph_id]
                        })
                
                # 3. Ensure top motifs are included
                all_motifs = data.groupby('motif')['logit_diff'].mean().sort_values(ascending=False)
                for motif, score in all_motifs.items():
                    if not any(m['motif'] == motif for m in structured_data):
                        motif_graphs = data[data['motif'] == motif]['graph_id'].unique().tolist()
                        structured_data.append({
                            'dataset': dataset,
                            'architecture': arch,
                            'model_type': model_type,
                            'motif': motif,
                            'avg_logit_diff': score,
                            'graph_ids': sorted(motif_graphs)
                        })
                        break  # Add at least one top motif if missing

                # Final processing
                df_structured = pd.DataFrame(structured_data)
                # df_structured = df_structured.drop_duplicates(subset=['motif', 'graph_ids'])
                df_structured = df_structured.sort_values(['avg_logit_diff', 'motif'], ascending=[False, True])
                
                # Save results
                filename = f"{structured_output_dir}/{dataset}_{arch}_{model_type}_enhanced.csv"
                df_structured.to_csv(filename, index=False)


In [ ]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "esol":["0205_ALL/0205esol", "0205_ALL/Vanilla_Reg"],
                  # "tox21":["0205_ALL/0205Tox",
                  #            "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type} (\d)
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold,architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold,architecture, model_type = match.groups()
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_path = os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoclr05_g(ent80)_p(sizebydatamodel2)_train_val_test.csv")
                    
                    dfs = []
                    try:
                        df = pd.read_csv(file_path)
                    except:
                        print(f"Unable to read: {file_path}")
                        continue
                            
                    dfs.append(df)
                    
                    # Only process further if we have at least one correct sample dataframe.
                    df_total = pd.concat(dfs, ignore_index=True)
                    
                    
                    # Create bins for sigmoid_importance (10 bins)
                    bin_edges = np.linspace(0, 1, 11)
                    # --- CALCULATE ABSOLUTE LOGIT DIFFERENCE ---
                    
                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_total['sigmoid_bin'] = pd.cut(df_total['sigmoid_importance'],
                                                           bins=bin_edges, include_lowest=True)
                    
                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_total)

# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )

# ================================
# PLOTTING: ONE PDF, EACH PAGE IS ONE ARCHITECTURE
# ================================
# Here the rows of the grid are datasets and the columns are unique types.
unique_datasets = sorted(root_dirs_dict.keys())
unique_types = types  # e.g., ["RBRICS", "MGSSL"]

# Determine grid dimensions:
n_datasets = len(unique_datasets)
n_arch = len(architectures)
n_cols_total = n_datasets * 2  # Two sub-columns per dataset, one per type.
n_rows_total = n_arch  # Rows correspond to architectures.

# Initialize legend handles & labels
legend_handles, legend_labels = None, None

for model_type in unique_types:
    # Determine grid dimensions: datasets as rows, architectures as columns.
    n_rows = len(unique_datasets)
    n_cols = len(architectures)
    
    # Create a figure with appropriate size.
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 8, n_rows * 6), squeeze=True)
    plt.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.1, wspace=0.25, hspace=0.35)
    
    legend_handles, legend_labels = None, None  # To capture legend once
    
    # Populate each subplot.
    for d_idx, dataset in enumerate(unique_datasets):
        for a_idx, arch in enumerate(architectures):
            ax = axes[d_idx, a_idx]
            # Retrieve data for this combination.
            data = plotting_data.get(arch, {}).get(dataset, {}).get(model_type, pd.DataFrame())
            
            if not data.empty:
                # Plotting code (similar to original).
                counts = data['sigmoid_bin'].value_counts().sort_index()
                counts_percentage = counts / counts.sum() if counts.sum() != 0 else pd.Series()
                # Aggregate data
                agg_data = data.groupby('sigmoid_bin').agg({
                    'gnnex_importance': 'mean',
                    'pgex_importance': 'mean'
                }).reset_index()

                # Melt for seaborn plotting
                melted = pd.melt(agg_data, id_vars=['sigmoid_bin'], 
                                value_vars=['gnnex_importance', 'pgex_importance'],
                                var_name='Method', value_name='Mean Importance')

                # Create plot on provided axis
                sns.barplot(x='sigmoid_bin', y='Mean Importance', hue='Method', 
                           data=melted, ax=ax, palette=['#1f77b4', '#ff7f0e'],
                           edgecolor='black')
#                 # Melt the data to combine importance scores from both methods
#                 melted = pd.melt(data, id_vars=['sigmoid_importance'], 
#                                  value_vars=['gnnex_importance', 'pgex_importance'],
#                                  var_name='Method', value_name='Importance')

#                 # Create scatter plot on provided axis
#                 sns.scatterplot(x='sigmoid_importance', y='Importance', hue='Method',
#                                 data=melted, ax=ax, palette=['#1f77b4', '#ff7f0e'],
#                                 alpha=0.6, edgecolor='none')

                # Customize axis
                ax.set_title('Explanation Method Comparison')
                ax.set_xlabel('Sigmoid Importance Bin')
                ax.set_ylabel('Average Importance Score')
                ax.tick_params(axis='x', rotation=45)
                ax.legend(title='Explanation Method')

            else:
                ax.set_visible(False)
    
    # Add dataset labels on the left.
    for d_idx, dataset in enumerate(unique_datasets):
        pos = axes[d_idx,0].get_position()
        fig.text(pos.x0 - 0.05, pos.y0 + pos.height/2, dataset, 
                 ha='right', va='center', rotation=90, fontsize=22, fontweight='bold')
    
    # Add architecture labels on top.
    for a_idx, arch in enumerate(["GAT", "GCN", "GIN"]):
        pos = axes[0,a_idx].get_position()
        fig.text(pos.x0 + pos.width/2, pos.y1 + 0.02, arch, 
                 ha='center', va='bottom', fontsize=22, fontweight='bold')
    
    # Add global legend.
    if legend_handles and legend_labels:
        fig.legend(
            legend_handles, 
            ['Vanilla', 'MOSE'], 
            title='Model', 
            loc='upper right', 
            bbox_to_anchor=(0.98, 0.93),  # Moves the legend to the top-right
            ncol=2,  # Keep items in a single row
            fontsize=14, 
            title_fontsize=16, 
            frameon=True, 
            edgecolor='black'
        )
    
    # Save the figure.
    plt.savefig(f'{model_type}_posthoc.png', bbox_inches='tight')
    plt.close()
print("Saved combined plots as combined_plots.png")


In [ ]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "esol":["0205_ALL/0205esol", "0205_ALL/Vanilla_Reg"],
                  # "tox21":["0205_ALL/0205Tox",
                  #            "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type} (\d)
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold,architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold,architecture, model_type = match.groups()
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_path = os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoclr05_g(ent80)_p(sizebydatamodel2)_train_val_test.csv")
                    
                    dfs = []
                    try:
                        df = pd.read_csv(file_path)
                    except:
                        print(f"Unable to read: {file_path}")
                        continue
                            
                    dfs.append(df)
                    
                    # Only process further if we have at least one correct sample dataframe.
                    df_total = pd.concat(dfs, ignore_index=True)
                    
                    
                    # Save the combined DataFrame in our nested dictionary.
                    plotting_data[architecture][dataset_name][model_type].append(df_total)

# Now, for each architecture, dataset, and type, combine the list of dataframes over all folds.
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            plotting_data[arch][dataset][model_type] = pd.concat(
                plotting_data[arch][dataset][model_type], ignore_index=True
            )


In [14]:
'''
With Vanilla weights loaded
'''
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import os
import json
import pandas as pd
from collections import defaultdict
import re
import pdb

# Set a high-quality style
sns.set_style("whitegrid")
sns.set_context("talk", font_scale=0.8)  # Larger fonts for readability


def Sigmoid(x):
    return 1/(1+np.exp(-x))

# Frequency threshold for motifs: only motifs that occur more than this value will be used.
motif_frequency_threshold = {
    "esol": 10,
    "BBBP": 18,
    "Lipophilicity": 38,
    "Mutagenicity": 70,
    "hERG": 90,
    "tox21":70,
}

root_dirs_dict = {
                  "Mutagenicity":["0205_ALL/0205Mutag",
                             "0205_ALL/Vanilla_BC"],
                  "hERG":["0205_ALL/0205herg",
                             "0205_ALL/Vanilla_BC"],
                  "Lipophilicity":["0205_ALL/0205Lipo",
                             "0205_ALL/Vanilla_Reg"],
                  "BBBP":["0205_ALL/0205BBBP",
                             "0205_ALL/Vanilla_BC"],
                  "esol":["0205_ALL/0205esol", "0205_ALL/Vanilla_Reg"],
                  # "tox21":["0205_ALL/0205Tox",
                  #            "0205_ALL/Vanilla_Tox"],
                 }


# Architectures and types to check
architectures = ["GATConv", "GCNConv", "GINConv"]
types = ["RBRICS"]

# Folder name pattern: EXPT-{number}R-{dataset}-{seed}-{fold}-{architecture}-{type} (\d)
folder_pattern = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(MGSSL|RBRICS)"
)

folder_pattern_vanilla = re.compile(
    r"EXPT-\d+[a-zA-Z]+-[a-zA-Z0-9]+-SEED-\d+-FOLD-(\d)+-(GATConv|GINConv|GCNConv)-.+-(None)?"
)
# ================================
# USER OPTIONS & SETTINGS
# ================================

# Optionally, if you wish to include incorrect samples, you can adjust this flag.
include_incorrect = True  
# Tolerance for considering a regression prediction “correct” (using logit vs. true value)
tolerance = 0.1  

# ================================
# COMBINING FOLD INFORMATION
# ================================
# We create a nested dictionary to collect data per architecture, per dataset, per type.
# Structure: plotting_data[architecture][dataset][model_type] will be a list of DataFrames (one per fold).
plotting_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

# Process each dataset.
for dataset_name, list_dirs in root_dirs_dict.items():
    for root_dir in list_dirs:
        # Loop over folders in this directory.
        for folder in os.listdir(root_dir):
            # Check if the folder matches one of our patterns.
            if folder_pattern.match(folder) or folder_pattern_vanilla.match(folder):
                # Extract fold, architecture, and model_type.
                match = folder_pattern.match(folder)
                if match is not None:
                    fold,architecture, model_type = match.groups()
                else:
                    match = folder_pattern_vanilla.match(folder)
                    fold,architecture, model_type = match.groups()
                
                # Only process if the architecture and type are among those of interest.
                if (architecture in architectures) and (model_type in types):
                    file_path = os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_withposthoclr05_g(ent80)_p(sizebydatamodel)_train_val_test.csv")
                    
                    file_paths = [
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_train.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_validation.csv"),
                        os.path.join(root_dir, folder, f"{dataset_name}_vanilla_impact_test.csv")
                    ]
                    dfs_logit_diff = []
                    for file in file_paths:
                        try:
                            
                            df = pd.read_csv(file)
                            df.rename(columns={'new_logit_class': 'new_logit'}, inplace=True)
                        except Exception as e:
                            print(f"Error reading {file}: {e}")
                            continue
                        dfs_logit_diff.append(df)
                        
                    if dfs_logit_diff:
                        df_total_logit_diff = pd.concat(dfs_logit_diff, ignore_index=True)
                    else:
                        continue
                            
                            
                    dfs = []
                    try:
                        df = pd.read_csv(file_path)
                    except:
                        print(f"Unable to read: {file_path}")
                        continue
                            
                    dfs.append(df)
                    
                    # Only process further if we have at least one correct sample dataframe.
                    df_total = pd.concat(dfs, ignore_index=True)

                    # Use the absolute difference between the sigmoid-transformed logits.
                    df_total_logit_diff['mose_impact'] =  np.abs(Sigmoid(df_total_logit_diff['original_logit']) - Sigmoid(df_total_logit_diff['new_logit']))
                    df_total_logit_diff['vanilla_impact'] =  np.abs(Sigmoid(df_total_logit_diff['original_logit_vanilla']) - Sigmoid(df_total_logit_diff['new_logit_vanilla']))
                    df_total_logit_diff['class_label'] = df_total_logit_diff['class_label']
                    
                    # Merge vanilla_impact and posthoc data on motif_id and motif
                    df_merged = pd.merge(
                        df_total_logit_diff,
                        df_total[['motif_id', 'motif', 'gnnex_importance', 'pgex_importance']],
                        on=['motif_id', 'motif'],
                        how='left'
                    )
                    

                    # Append the merged DataFrame instead of df_total
                    plotting_data[architecture][dataset_name][model_type].append(df_merged)

# After combining all folds and architectures, compute correlations:
correlation_results = []
for arch in plotting_data:
    for dataset in plotting_data[arch]:
        for model_type in plotting_data[arch][dataset]:
            df_list = plotting_data[arch][dataset][model_type]
            df = pd.concat(
                df_list, ignore_index=True
            )
            if not isinstance(df, pd.DataFrame) or df.empty:
                print(f"Skipping {arch}/{dataset}/{model_type}: invalid data")
                continue
            # Calculate correlations (handle NaN)
            c1 = df['mose_impact'].corr(df['sigmoid_importance'], method='pearson')
            c2 = df['vanilla_impact'].corr(df['sigmoid_importance'], method='pearson')
            c3 = df['vanilla_impact'].corr(df['gnnex_importance'], method='pearson')
            c4 = df['vanilla_impact'].corr(df['pgex_importance'], method='pearson')
            if np.isnan(c4).any():
                print(df['vanilla_impact'],df['pgex_importance'],"value",c4)

            correlation_results.append({
                'architecture': arch,
                'dataset': dataset,
                'model_type': model_type,
                'C1_mose_vs_sigmoid': c1,
                'C2_vanilla_vs_sigmoid': c2,
                'C3_vanilla_vs_gnnex': c3,
                'C4_vanilla_vs_pgex': c4
            })

# Convert results to DataFrame and save
correlation_df = pd.DataFrame(correlation_results)
correlation_df.to_csv("correlation_results_pearson.csv", index=False)
            
#Todo calculate C1= corelation between mose_impact and sigmoid_importance , C2=corelation between vanilla_impact and sigmoid_importance, C2=corelation between vanilla_impact and gnnex_importance, C4=corelation between vanilla_impact and pgex_importance


Error reading 0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_train.csv: [Errno 2] No such file or directory: '0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_train.csv'
Error reading 0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_validation.csv: [Errno 2] No such file or directory: '0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_validation.csv'
Error reading 0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_test.csv: [Errno 2] No such file or directory: '0205_ALL/0205Mutag/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GATConv-SingleChannel-RBRICS/Mutagenicity_vanilla_impact_test.csv'
Error reading 0205_ALL/0205herg/EXPT-12BC-Mutagenicity-SEED-0-FOLD-4-GCNConv-SingleChannel-R